# 12 · Full champion / challenger pipeline

Reproduce the build-plan architecture **node-by-node** for the **v2 challenger**, one ClearML step per major stage:

1. **Prepare data** — load raw v2 splits (train / validation / test)
2. **Feature engineering** — transaction + behavioural features
3. **Logistic regression** — baseline
4. **LightGBM** — baseline candidate
5. **Select winner (HITL)** — compare F1/ROC-AUC/PR-AUC (higher better), human picks
6. **Finalize challenger** — Optuna HPO + retrain if LightGBM won, else ship logistic
7. **Champion (v1) / Challenger (v2) comparison** — widened metrics + agreement/JSD/PSI + HITL promotion

v1 is the **production champion** (already deployed, loaded from artifacts) and appears only for the final comparison. The final decision is based on quality + agreement/PSI drift, with human confirmation at both the model-selection and promotion gates.

Steps run **locally, in-process** (`run_pipeline_steps_locally=True`) so no ClearML agent/queue is required.

In [59]:
from cross_model_drift.config import load_config

config = load_config()
config.clearml_project, config.clearml_web_host, config.clearml_api_host, config.classification_threshold

# Set True to force every cached step to rebuild (ignore existing artifacts).
FORCE_RERUN = False

## Pipeline nodes — one per architecture stage

```text
prepare_v2_data
       │
       ▼
engineer_v2_features
       │
       ├────────────→ train_v2_logistic_baseline ─────┐
       └────────────→ train_v2_lightgbm_baseline ─────┤  (parallel tracks)
                                                      ▼
                                              select_v2_model
                                       (F1/ROC-AUC/PR-AUC, suggest + HITL)
                                                      │
                                                      ▼
                                        finalize_v2_challenger
                               (HPO + retrain if LightGBM won, else logistic)
                                                      │
                                                      ▼
                             compare_v1_champion_v2  ← loads saved v1 champion
                  (widened metrics + agreement + JSD + PSI + HITL promotion)
```

Each node is a ClearML function-step. Both model baselines run in parallel, then a
human picks the winner (F1/ROC-AUC/PR-AUC, higher better) before tuning. The final
comparison suggests a promotion only when agreement > 95%, JSD < 0.05, PSI < 0.1,
then asks the human to confirm.

In [60]:
from clearml import PipelineController

In [61]:
def _load_json(path) -> dict:
    import json

    return json.loads(path.read_text())

In [62]:
from cross_model_drift.splits import ModelVersion


def prepare_v2_data(version: ModelVersion, force_rerun: bool = False) -> dict[str, int]:
    """Stage 1 · Prepare data — load raw v2 splits (no feature engineering)."""
    from pathlib import Path

    from cross_model_drift.config import load_config
    from cross_model_drift.data import load_split

    cfg = load_config()
    cache = cfg.artifacts_path() / "pipeline" / f"{version}_data_sizes.json"
    if cache.exists() and not force_rerun:
        import json

        return json.loads(cache.read_text())
    sizes = {
        f"n_{split}_rows": len(load_split(version, split, cfg, engineer=False))
        for split in ("train", "validation", "test")
    }
    cache.parent.mkdir(parents=True, exist_ok=True)
    import json

    cache.write_text(json.dumps(sizes))
    return sizes


def engineer_v2_features(version: ModelVersion, force_rerun: bool = False) -> dict[str, int]:
    """Stage 2 · Feature engineering — transaction + behavioural features."""
    from pathlib import Path

    from cross_model_drift.config import load_config
    from cross_model_drift.data import load_split

    cfg = load_config()
    cache = cfg.artifacts_path() / "pipeline" / f"{version}_featured_sizes.json"
    if cache.exists() and not force_rerun:
        import json

        return json.loads(cache.read_text())
    sizes = {
        f"n_{split}_rows": len(load_split(version, split, cfg, engineer=True))
        for split in ("train", "validation", "test")
    }
    cache.parent.mkdir(parents=True, exist_ok=True)
    import json

    cache.write_text(json.dumps(sizes))
    return sizes

In [63]:
from cross_model_drift.splits import ModelVersion


def train_v2_logistic_baseline(version: ModelVersion, force_rerun: bool = False) -> dict[str, float]:
    from pathlib import Path

    from cross_model_drift.config import load_config
    from cross_model_drift.models import train_logistic

    cfg = load_config()
    threshold = cfg.classification_threshold
    root = cfg.artifacts_path() / "models"
    path = root / f"{version}_pipeline_logistic.joblib"
    sidecar = root / f"{version}_pipeline_logistic.json"

    if not path.exists() or force_rerun:
        from cross_model_drift.data import load_split
        from cross_model_drift.features import target_vector
        from cross_model_drift.metrics import quality_metrics

        train = load_split(version, "train", cfg)
        valid = load_split(version, "validation", cfg)
        model = train_logistic(train, target_vector(train), threshold=threshold)
        model.save(path)
        m = quality_metrics(target_vector(valid), model.predict_proba(valid), threshold=threshold)
        metrics = {
            "precision": m["precision"],
            "recall": m["recall"],
            "f1": m["f1"],
            "roc_auc": m["roc_auc"],
            "pr_auc": m["pr_auc"],
        }
        import json

        root.mkdir(parents=True, exist_ok=True)
        sidecar.write_text(json.dumps(metrics))
    else:
        import json

        cached = json.loads(sidecar.read_text())
        metrics = {k: float(v) for k, v in cached.items() if k in ("precision", "recall", "f1", "roc_auc", "pr_auc")}
    return metrics

In [64]:
from cross_model_drift.splits import ModelVersion


def train_v2_lightgbm_baseline(version: ModelVersion, force_rerun: bool = False) -> dict[str, float]:
    """Stage 4 · LightGBM baseline — untuned candidate, scored on v2 validation."""
    from pathlib import Path

    from cross_model_drift.config import load_config
    from cross_model_drift.models import train_lightgbm

    cfg = load_config()
    threshold = cfg.classification_threshold
    root = cfg.artifacts_path() / "models"
    path = root / f"{version}_lightgbm.joblib"
    sidecar = root / f"{version}_lightgbm.json"

    if not path.exists() or force_rerun:
        from cross_model_drift.data import load_split
        from cross_model_drift.features import target_vector
        from cross_model_drift.metrics import quality_metrics

        train = load_split(version, "train", cfg)
        valid = load_split(version, "validation", cfg)
        model = train_lightgbm(
            train, target_vector(train), valid, target_vector(valid), threshold=threshold,
        )
        model.save(path)
        m = quality_metrics(target_vector(valid), model.predict_proba(valid), threshold=threshold)
        metrics = {
            "precision": m["precision"],
            "recall": m["recall"],
            "f1": m["f1"],
            "roc_auc": m["roc_auc"],
            "pr_auc": m["pr_auc"],
        }
        import json

        root.mkdir(parents=True, exist_ok=True)
        sidecar.write_text(json.dumps(metrics))
    else:
        import json

        cached = json.loads(sidecar.read_text())
        metrics = {k: float(v) for k, v in cached.items() if k in ("precision", "recall", "f1", "roc_auc", "pr_auc")}
    return metrics

In [65]:
from cross_model_drift.splits import ModelVersion


def select_v2_model(
    version: ModelVersion,
    decision: str = "",
) -> dict[str, object]:
    """Select the model to carry on as the v2 challenger (F1/ROC-AUC/PR-AUC, higher better).

    Reads the baseline metric sidecars written by the two parallel baseline steps
    (so this step is self-contained and does not depend on serialized dict kwargs).

    Created as a DRAFT. The pipeline pauses after the parallel baselines; review the
    metrics printed here, then edit the `decision` kwarg (logistic | lightgbm) in the
    ClearML UI and run the draft to continue.
    """
    import json

    from cross_model_drift.config import load_config

    cfg = load_config()
    root = cfg.artifacts_path() / "models"
    logistic = json.loads((root / f"{version}_pipeline_logistic.json").read_text())
    lightgbm = json.loads((root / f"{version}_lightgbm.json").read_text())

    import pandas as pd

    df = pd.DataFrame(
        {
            "metric": ["f1", "roc_auc", "pr_auc"],
            "logistic": [logistic["f1"], logistic["roc_auc"], logistic["pr_auc"]],
            "lightgbm": [lightgbm["f1"], lightgbm["roc_auc"], lightgbm["pr_auc"]],
            "winner": [
                "logistic" if logistic["f1"] >= lightgbm["f1"] else "lightgbm",
                "logistic" if logistic["roc_auc"] >= lightgbm["roc_auc"] else "lightgbm",
                "logistic" if logistic["pr_auc"] >= lightgbm["pr_auc"] else "lightgbm",
            ],
        }
    )
    wins = df["winner"].value_counts()
    suggestion = "logistic" if wins.get("logistic", 0) > wins.get("lightgbm", 0) else "lightgbm"

    print("Validation metrics (higher is better for F1, ROC-AUC, PR-AUC):")
    print(df.to_string(index=False))
    print(f"\nWinner suggestion: {suggestion}")

    choice = (decision or "").strip().lower()
    if choice not in ("logistic", "lightgbm"):
        choice = suggestion

    result: dict[str, object] = {"winner": choice, "suggestion": suggestion}
    root.mkdir(parents=True, exist_ok=True)
    (root / f"{version}_selected_model.json").write_text(json.dumps(result))
    return result

In [66]:
from cross_model_drift.splits import ModelVersion


def hpo_v2_challenger(
    version: ModelVersion,
    n_trials: int = 10,
    force_rerun: bool = False,
) -> dict[str, object]:
    """HPO node · Tune the confirmed winner for the v2 challenger.

    Reads the winner from the `v2_selected_model.json` written by select_v2_model,
    so this step is self-contained (no serialized dict kwargs).

    - LightGBM: run Optuna HPO, retrain with best params, save tuned challenger, score on v2 test.
    - Logistic: no HPO (nothing to tune); shortcut to the saved logistic baseline as the challenger.
    """
    import json

    from cross_model_drift.config import load_config

    cfg = load_config()
    threshold = cfg.classification_threshold
    root = cfg.artifacts_path() / "models"
    winner = str(json.loads((root / f"{version}_selected_model.json").read_text())["winner"])

    if winner == "logistic":
        path = root / f"{version}_pipeline_logistic.joblib"
        sidecar = root / f"{version}_pipeline_logistic.json"
        metrics = json.loads(sidecar.read_text())
        return {"challenger_path": str(path), "winner": "logistic", **metrics}

    # ---- LightGBM: HPO then retrain ----#
    from cross_model_drift.data import load_split
    from cross_model_drift.features import target_vector
    from cross_model_drift.hpo import run_optuna_hpo
    from cross_model_drift.metrics import quality_metrics
    from cross_model_drift.models import train_lightgbm

    best_path = root / f"{version}_best_params.json"
    tuned_path = root / f"{version}_challenger_tuned.joblib"
    sidecar = root / f"{version}_challenger_tuned.json"
    if (
        all(p.exists() for p in (best_path, tuned_path, sidecar))
        and not force_rerun
    ):
        metrics = json.loads(sidecar.read_text())
        return {"challenger_path": str(tuned_path), "winner": "lightgbm", **metrics}

    train = load_split(version, "train", cfg)
    valid = load_split(version, "validation", cfg)
    test = load_split(version, "test", cfg)
    best_params = json.loads(best_path.read_text()) if best_path.exists() else None
    if best_params is None:
        study = run_optuna_hpo(train, valid, n_trials=n_trials, threshold=threshold)
        best_params = dict(study.best_params)
        best_params["best_pr_auc"] = float(study.best_value)
        root.mkdir(parents=True, exist_ok=True)
        best_path.write_text(json.dumps(best_params))

    model = train_lightgbm(
        train, target_vector(train), valid, target_vector(valid),
        params=best_params, threshold=threshold,
    )
    model.save(tuned_path)
    m = quality_metrics(target_vector(test), model.predict_proba(test), threshold=threshold)
    metrics = {
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "roc_auc": m["roc_auc"],
        "pr_auc": m["pr_auc"],
        "best_pr_auc": best_params.get("best_pr_auc", 0.0),
    }
    root.mkdir(parents=True, exist_ok=True)
    sidecar.write_text(json.dumps(metrics))
    return {"challenger_path": str(tuned_path), "winner": "lightgbm", **metrics}

In [67]:
from cross_model_drift.splits import ModelVersion


def compare_v1_champion_v2(
    v1_version: ModelVersion,
    v2_version: ModelVersion,
) -> dict[str, float]:
    """Stage 7 · Champion (v1) vs Challenger (v2) on the same holdout.

    Widened metrics: Precision, Recall, F1, ROC-AUC, PR-AUC per model +
    agreement, JSD, PSI. Suggests promotion when agreement > 95%, JSD < 0.05,
    PSI < 0.1. No HITL here — the confirm step does that. Writes the comparison
    to a sidecar JSON so the confirm draft step can read it without serialized kwargs.
    """
    import json

    from cross_model_drift.compare import compare_models
    from cross_model_drift.config import load_config
    from cross_model_drift.data import load_split
    from cross_model_drift.models import load_model

    cfg = load_config()
    threshold = cfg.classification_threshold
    root = cfg.artifacts_path() / "models"
    v1 = load_model(root / "v1_champion_challenger.joblib")
    v2 = load_model(root / f"{v2_version}_challenger_tuned.joblib")
    reference = load_split(v1_version, "train", cfg)
    holdout = load_split(v1_version, "holdout", cfg)
    result = compare_models(holdout, v1, v2, reference=reference, threshold=threshold)

    import pandas as pd

    metrics = pd.DataFrame(
        {
            "metric": ["precision", "recall", "f1", "roc_auc", "pr_auc"],
            "v1": [result.v1_metrics["precision"], result.v1_metrics["recall"], result.v1_metrics["f1"], result.v1_metrics["roc_auc"], result.v1_metrics["pr_auc"]],
            "v2": [result.v2_metrics["precision"], result.v2_metrics["recall"], result.v2_metrics["f1"], result.v2_metrics["roc_auc"], result.v2_metrics["pr_auc"]],
        }
    )
    agreement = float(result.agreement["agreement"])
    score_jsd = float(result.score_jsd)
    psi_max = float(result.psi["psi"].max())

    suggest_promote = 1.0 if (agreement > 0.95 and score_jsd < 0.05 and psi_max < 0.1) else 0.0

    print("Holdout metrics (v1 champion vs v2 challenger):")
    print(metrics.to_string(index=False))
    print(f"\nAgreement: {agreement:.4f}")
    print(f"Score JSD: {score_jsd:.4f}")
    print(f"Max PSI:   {psi_max:.4f}")
    print(f"\nPromotion suggestion: {'PROMOTE v2' if suggest_promote else 'DO NOT PROMOTE v2'}")

    comparison = {
        "agreement": agreement,
        "score_jsd": score_jsd,
        "psi_max": psi_max,
        "suggestion_promote": suggest_promote,
        "v1_pr_auc": float(result.v1_metrics["pr_auc"]),
        "v2_pr_auc": float(result.v2_metrics["pr_auc"]),
    }
    root.mkdir(parents=True, exist_ok=True)
    (root / f"{v2_version}_comparison.json").write_text(json.dumps(comparison))
    return comparison

In [68]:
from cross_model_drift.splits import ModelVersion


def confirm_v2_model(
    comparison: dict[str, float],
    promote: str = "",
) -> dict[str, object]:
    """HITL confirm · Promote v2 or not, given the comparison suggestion.

    This step is created as a DRAFT. Review the comparison summary printed here, then
    edit the `promote` kwarg in the ClearML UI ("true" | "false") and run the draft to
    finish. Leave it empty to follow the suggestion.
    """
    agreement = comparison["agreement"]
    score_jsd = comparison["score_jsd"]
    psi_max = comparison["psi_max"]
    suggest_promote = comparison["suggestion_promote"] > 0

    print("Comparison summary:")
    print(f"  Agreement: {agreement:.4f} (suggest condition: > 0.95)")
    print(f"  Score JSD: {score_jsd:.4f} (suggest condition: < 0.05)")
    print(f"  Max PSI:   {psi_max:.4f} (suggest condition: < 0.1)")
    print(f"  Suggested: {'PROMOTE v2' if suggest_promote else 'DO NOT PROMOTE v2'}")

    override = (promote or "").strip().lower()
    if override in ("true", "1", "yes", "y"):
        final_promote = True
    elif override in ("false", "0", "no", "n"):
        final_promote = False
    else:
        final_promote = suggest_promote

    return {
        "agreement": agreement,
        "score_jsd": score_jsd,
        "psi_max": psi_max,
        "suggestion_promote": suggest_promote,
        "promote_v2": final_promote,
    }

## Assemble and start the pipeline

The `select_v2_model` node waits for **both** parallel baselines, then the pipeline **pauses**: it's created as a **draft** step. In the ClearML UI, review the printed metrics and set the `decision` kwarg (logistic | lightgbm), then run the draft. `hpo_v2_challenger` tunes the winner (HPO only if LightGBM won; logistic shortcuts through). `compare_v1_champion_v2` loads the saved v1 champion and computes the widened comparison. `confirm_v2_model` is the second **draft** step: review the summary and set `promote` (true | false), then run it to finish. Steps run **locally**.

In [ ]:
pipe = PipelineController(
    name="champion-challenger-full-v2",
    project=config.clearml_project,
    version="0.5.0",
    add_pipeline_tags=False,
)

# Stage 1 · Prepare data
pipe.add_function_step(
    name="prepare_v2_data",
    function=prepare_v2_data,
    function_kwargs={"version": "v2", "force_rerun": FORCE_RERUN},
    function_return=["sizes"],
)
# Stage 2 · Feature engineering
pipe.add_function_step(
    name="engineer_v2_features",
    function=engineer_v2_features,
    function_kwargs={"version": "v2", "force_rerun": FORCE_RERUN},
    function_return=["featured_sizes"],
    parents=["prepare_v2_data"],
)
# Stage 3 · Logistic regression baseline (parallel track 1)
pipe.add_function_step(
    name="train_v2_logistic_baseline",
    function=train_v2_logistic_baseline,
    function_kwargs={"version": "v2", "force_rerun": FORCE_RERUN},
    function_return=["logistic_metrics"],
    parents=["engineer_v2_features"],
)
# Stage 4 · LightGBM baseline (parallel track 2)
pipe.add_function_step(
    name="train_v2_lightgbm_baseline",
    function=train_v2_lightgbm_baseline,
    function_kwargs={"version": "v2", "force_rerun": FORCE_RERUN},
    function_return=["lgbm_metrics"],
    parents=["engineer_v2_features"],
)
# Stage 4·5 · Select winner (HITL) — depends on BOTH parallel baselines
pipe.add_function_step(
    name="select_v2_model",
    function=select_v2_model,
    function_kwargs={"version": "v2", "decision": "lightgbm"},
    function_return=["selection"],
    parents=["train_v2_logistic_baseline", "train_v2_lightgbm_baseline"],
)
# Stage 5·6 · HPO (tune model; logistic shortcut if it won)
pipe.add_function_step(
    name="hpo_v2_challenger",
    function=hpo_v2_challenger,
    function_kwargs={"version": "v2", "n_trials": 10, "force_rerun": FORCE_RERUN},
    function_return=["challenger"],
    parents=["select_v2_model"],
)
# Stage 7 · Champion (v1) vs challenger (v2) comparison (no HITL)
pipe.add_function_step(
    name="compare_v1_champion_v2",
    function=compare_v1_champion_v2,
    function_kwargs={"v1_version": "v1", "v2_version": "v2"},
    function_return=["comparison"],
    parents=["hpo_v2_challenger"],
)
# Stage 8 · HITL confirm promotion (dedicated draft node)
pipe.add_function_step(
    name="confirm_v2_model",
    function=confirm_v2_model,
    function_kwargs={
        "comparison": "${compare_v1_champion_v2.comparison}",
        "promote": "",
    },
    function_return=["decision"],
    parents=["compare_v1_champion_v2"],
)

# Steps run in-process (no agent/queue needed). Cached steps reuse artifacts when
# they exist; set FORCE_RERUN=True to rebuild everything. select_v2_model and
# confirm_v2_model are DRAFT steps: the pipeline pauses at each, and you edit their
# kwargs (decision / promote) in the ClearML UI, then run the draft to continue.
pipe.start_locally(run_pipeline_steps_locally=True)
"pipeline defined and started locally"

Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module


ClearML Task: created new task id=80dd152666624597bb8c0f3e3d520995
ClearML results page: http://localhost:8080/projects/31e8b28eb97e4e2d96a6c91dfab274b9/tasks/80dd152666624597bb8c0f3e3d520995/output/log
ClearML pipeline page: http://localhost:8080/pipelines/31e8b28eb97e4e2d96a6c91dfab274b9/experiments/80dd152666624597bb8c0f3e3d520995


Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module
Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module
Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module
Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module
Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module
Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module
Could not fetch functi

Launching the next 1 steps
Launching step [prepare_v2_data]
Launching step: prepare_v2_data
Parameters:
{'kwargs/version': 'v2', 'kwargs/force_rerun': False}
Configurations:
{}
Overrides:
{}
ClearML results page: http://localhost:8080/projects/31e8b28eb97e4e2d96a6c91dfab274b9/tasks/b296c4d6609645249b38fc7edbf72449/output/log
2026-08-24 00:11:08,073 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found
Launching the next 1 steps
Launching step [engineer_v2_features]
Launching step: engineer_v2_features
Parameters:
{'kwargs/version': 'v2', 'kwargs/force_rerun': False}
Configurations:
{}
Overrides:
{}
ClearML results page: http://localhost:8080/projects/31e8b28eb97e4e2d96a6c91dfab274b9/tasks/de4d1843d2de45c39345c9a46a78d96a/output/log
2026-08-24 00:11:32,277 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found
Launching the next 2 steps
Launching step [train_v2_lightgbm_baseline]
Launching step [train_v2_